In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Admin\Desktop\Nanoparticle-toxicity\dataset\nanotox_dataset.csv")

In [2]:
print(df.shape)

(881, 11)


In [3]:
print(df.head())

     NPs  coresize  hydrosize  surfcharge  surfarea    Ec  Expotime  dosage  \
0  Al2O3      39.7      267.0        36.3      64.7 -1.51        24   0.001   
1  Al2O3      39.7      267.0        36.3      64.7 -1.51        24   0.010   
2  Al2O3      39.7      267.0        36.3      64.7 -1.51        24   0.100   
3  Al2O3      39.7      267.0        36.3      64.7 -1.51        24   1.000   
4  Al2O3      39.7      267.0        36.3      64.7 -1.51        24   5.000   

      e  NOxygen     class  
0  1.61        3  nonToxic  
1  1.61        3  nonToxic  
2  1.61        3  nonToxic  
3  1.61        3  nonToxic  
4  1.61        3  nonToxic  


In [4]:
print(df.dtypes)

NPs               str
coresize      float64
hydrosize     float64
surfcharge    float64
surfarea      float64
Ec            float64
Expotime        int64
dosage        float64
e             float64
NOxygen         int64
class             str
dtype: object


In [6]:
print(df['class'].value_counts())

class
Toxic       476
nonToxic    405
Name: count, dtype: int64


In [7]:
print(df['NPs'].value_counts())

NPs
ZnO      594
TiO2     200
CuO       51
Al2O3     18
Fe2O3     18
Name: count, dtype: int64


In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [11]:
df['NPs_encoded'] = le.fit_transform(df['NPs'])

In [12]:
X = df.drop(columns=['NPs','class'])
y = df['class'].map({'Toxic':1, 'nonToxic':0})

In [13]:
print(X.shape)
print(X.columns.tolist())
print(y.value_counts())

(881, 10)
['coresize', 'hydrosize', 'surfcharge', 'surfarea', 'Ec', 'Expotime', 'dosage', 'e', 'NOxygen', 'NPs_encoded']
class
1    476
0    405
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [17]:
print('Train size:', X_train.shape)
print('Test size:', X_test.shape)
print('Train class balance:', y_train.value_counts().to_dict())
print('Test class balance:', y_test.value_counts().to_dict())

Train size: (704, 10)
Test size: (177, 10)
Train class balance: {1: 380, 0: 324}
Test class balance: {1: 96, 0: 81}


In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# re-fit scaler on DataFrame to preserve feature names
scaler = StandardScaler()
scaler.fit(X_train)
joblib.dump(scaler, 'scaler.pkl')


['scaler.pkl']

In [19]:
print('Mean of first feature (should be 0):', X_train_scaled[:, 0].mean().round(4))
print('Std of first feature (should be 1):', X_train_scaled[:, 0].std().round(4))

Mean of first feature (should be 0): -0.0
Std of first feature (should be 1): 1.0


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

rf=RandomForestClassifier(n_estimators=100, random_state=42)
svm=SVC(kernel='rbf',probability=True, random_state=42)
knn=KNeighborsClassifier(n_neighbors=5)

rf.fit(X_train_scaled, y_train)
svm.fit(X_train_scaled,y_train)
knn.fit(X_train_scaled, y_train)

print('All 3 Models trained.')

All 3 Models trained.


In [27]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import cross_val_score


In [29]:
models = {'Random Forest': rf, 'SVM': svm, 'KNN': knn}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cv = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()
    cm = confusion_matrix(y_test, y_pred)
    
    print(f'--- {name} ---')
    print(f'Accuracy:         {acc:.4f}')
    print(f'ROC-AUC:          {auc:.4f}')
    print(f'Cross-val (5-fold): {cv:.4f}')
    print(f'Confusion Matrix:\n{cm}')
    print()

--- Random Forest ---
Accuracy:         0.9887
ROC-AUC:          0.9995
Cross-val (5-fold): 0.9645
Confusion Matrix:
[[79  2]
 [ 0 96]]

--- SVM ---
Accuracy:         0.9266
ROC-AUC:          0.9856
Cross-val (5-fold): 0.9176
Confusion Matrix:
[[69 12]
 [ 1 95]]

--- KNN ---
Accuracy:         0.9492
ROC-AUC:          0.9915
Cross-val (5-fold): 0.9006
Confusion Matrix:
[[78  3]
 [ 6 90]]



In [30]:
import joblib

joblib.dump(rf, 'best_model_rf.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

print('Model, scaler, and label encoder saved.')

Model, scaler, and label encoder saved.
